# ZTF Image Download

Downloads ZTF difference-image, science-image, and reference-image cutouts for galaxies in a localization volume, matches them back to a galaxy catalog, and saves them to per-galaxy folders.

In [14]:
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord

from ztf_downloads.ztf_search import metadata_search_resumable
from ztf_downloads.ztf_download import (
    build_sci_url,
    build_ref_url,
    build_cutout_url,
    batch_download_resumable,
    _safe_label,
)

## 1. Load galaxy catalog

In [15]:
galaxies = pd.read_csv(
    'ZTF22aabjpxh_galaxy_catalog.csv',
    low_memory=False,
).copy()

# 1-based row number used as the per-galaxy folder name
galaxies['galaxy_row'] = galaxies.index + 1

# Coordinates of all galaxies (used for sky-matching later)
gal_coords = SkyCoord(
    ra=galaxies['ra_fin'].to_numpy() * u.deg,
    dec=galaxies['dec_fin'].to_numpy() * u.deg,
)

# Plain (ra, dec) tuples, in the format metadata_search_resumable expects
coords = list(zip(galaxies['ra_fin'], galaxies['dec_fin']))

print(f"{len(galaxies)} galaxies in localization volume")

46365 galaxies in localization volume


## 2. Download image metadata (can be used to download sci, diff, ref)

In [ ]:
diff_meta_file = "meta_test.csv"
diff_meta_progress = "test.progress.json"

df_meta = metadata_search_resumable(
    positions=coords,
    output_csv=diff_meta_file,
    progress_json=diff_meta_progress,
    batch_size=100,
    size_deg=0.01,
    date_start="2022-02-19",  # TODO: derive from event trigger time
    date_end="2022-02-21",    # TODO: derive as date_start + 2 days
    timeout=300,
)

print(f"Current metadata rows in {diff_meta_file}: {len(df_meta)}")

Metadata batch 1/464 (0:100) attempt 1
Metadata batch 2/464 (100:200) attempt 1


KeyboardInterrupt: 

In [ ]:
# If resuming a previous run, skip the metadata download above and load the cached CSV instead:
diff_meta_file = "meta_test.csv"
df_meta = pd.read_csv(diff_meta_file)
print(f"Loaded rows: {len(df_meta)}")

Loaded rows: 313


## 3. Download difference images (filters zr, zi)

> ! For ~70k images this can take ~16 hours and download ~40 GB. 

In [18]:
# Keep only the r/i filters
allowed_filters = {"zr", "zi"}
df_meta = df_meta[df_meta["filtercode"].isin(allowed_filters)].copy()
print(f"Rows after filter restriction (zr, zi): {len(df_meta)}")

# Match each difference-image row to its nearest galaxy
meta_coords = SkyCoord(
    ra=df_meta["in_ra"].to_numpy() * u.deg,
    dec=df_meta["in_dec"].to_numpy() * u.deg,
)
match_idx, sep2d, _ = meta_coords.match_to_catalog_sky(gal_coords)
tol = 2.0 * u.arcsec
is_match = sep2d <= tol

# Assign the galaxy's row number (= folder name) to matched rows
matched_folders = galaxies.iloc[match_idx]["galaxy_row"].astype(str).to_numpy()
df_meta["gal_folder"] = pd.Series(
    np.where(is_match, matched_folders, "unmatched"),
    index=df_meta.index,
)
df_meta["gal_folder"] = df_meta["gal_folder"].apply(_safe_label)

print(f"df_meta length: {len(df_meta)}")
print(f"Rows with galaxy match (<= {tol}): {is_match.sum()}")
print(f"Unmatched rows: {(~is_match).sum()}")
print(f"Max separation: {sep2d.max().to(u.arcsec):.3f}")

Rows after filter restriction (zr, zi): 239
df_meta length: 239
Rows with galaxy match (<= 2.0 arcsec): 239
Unmatched rows: 0
Max separation: 0.000 arcsec


In [19]:
# Build one (url, filename, folder) tuple per download task
cutout_inputs = []
for _, row in df_meta.iterrows():
    base_sci_url = build_sci_url(row, suffix="scimrefdiffimg.fits.fz")
    original_fname = Path(urlparse(base_sci_url).path).name
    url = build_cutout_url(base_sci_url, row["in_ra"], row["in_dec"], size_arcsec=240)

    filefracday_str = str(int(row["filefracday"])).zfill(14)
    custom_label = f"RA{row['in_ra']:.4f}_DEC{row['in_dec']:.4f}_{row['filtercode']}_{filefracday_str}"
    desired_fname = f"{row['gal_folder']}_{custom_label}__{original_fname}"

    cutout_inputs.append((url, desired_fname, row['gal_folder']))

In [20]:
batch_download_resumable(
    cutout_inputs,
    out_dir="diff_test",       # parent directory
    progress_json="test.progress.json",
    max_workers=9,
    chunk_size=200,
)

# List all downloaded FITS files (nested in per-galaxy subfolders)
diff_cutouts = sorted(str(p) for p in Path("diff_test").rglob("*.fits*"))
print(f"Diff files currently on disk: {len(diff_cutouts)}")

Download chunk 1/2 (0:200) attempt 1
1 Downloaded: 3_RA240.0356_DEC31.5785_zr_20220220510833__ztf_20220220510833_000679_zr_c08_o_q3_scimrefdiffimg.fits.fz
2 Downloaded: 1_RA240.0348_DEC31.5102_zr_20220220510833__ztf_20220220510833_000679_zr_c08_o_q3_scimrefdiffimg.fits.fz
3 Downloaded: 2_RA240.0356_DEC31.4340_zr_20220219500093__ztf_20220219500093_001676_zr_c10_o_q1_scimrefdiffimg.fits.fz
4 Downloaded: 3_RA240.0356_DEC31.5785_zi_20220219536262__ztf_20220219536262_000679_zi_c08_o_q3_scimrefdiffimg.fits.fz
5 Downloaded: 8_RA240.0365_DEC31.6018_zi_20220219536262__ztf_20220219536262_000679_zi_c08_o_q3_scimrefdiffimg.fits.fz
6 Downloaded: 1_RA240.0348_DEC31.5102_zi_20220219536262__ztf_20220219536262_000679_zi_c08_o_q3_scimrefdiffimg.fits.fz
7 Downloaded: 1_RA240.0348_DEC31.5102_zr_20220219500093__ztf_20220219500093_001676_zr_c10_o_q1_scimrefdiffimg.fits.fz
8 Downloaded: 6_RA240.0363_DEC31.3740_zr_20220219500093__ztf_20220219500093_001676_zr_c10_o_q1_scimrefdiffimg.fits.fz
9 Downloaded: 4_RA2

## 4. Download science images

Same detections as the difference images above — reuses `df_meta`.

In [21]:
sci_inputs = []
for _, row in df_meta.iterrows():
    base_sci_url = build_sci_url(row, suffix="sciimg.fits")
    original_fname = Path(urlparse(base_sci_url).path).name
    url = build_cutout_url(base_sci_url, row["in_ra"], row["in_dec"], size_arcsec=240)

    filefracday_str = str(int(row["filefracday"])).zfill(14)
    custom_label = f"RA{row['in_ra']:.4f}_DEC{row['in_dec']:.4f}_{row['filtercode']}_{filefracday_str}"
    desired_fname = f"{row['gal_folder']}_{custom_label}__{original_fname}"

    sci_inputs.append((url, desired_fname, row['gal_folder']))

batch_download_resumable(
    sci_inputs,
    out_dir="sci_test",
    progress_json="sci_test.json",
    max_workers=9,
    chunk_size=200,
)

sci_cutouts = sorted(str(p) for p in Path("sci_test").rglob("*.fits*"))
print(f"Science files currently on disk: {len(sci_cutouts)}")

Download chunk 1/2 (0:200) attempt 1
1 Downloaded: 1_RA240.0348_DEC31.5102_zr_20220219500093__ztf_20220219500093_001676_zr_c10_o_q1_sciimg.fits
2 Downloaded: 1_RA240.0348_DEC31.5102_zr_20220220510833__ztf_20220220510833_000679_zr_c08_o_q3_sciimg.fits
3 Downloaded: 1_RA240.0348_DEC31.5102_zi_20220219536262__ztf_20220219536262_000679_zi_c08_o_q3_sciimg.fits
Retry 1/3 for https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci/2022/0220/510833/ztf_20220220510833_000679_zr_c08_o_q3_sciimg.fits?center=240.03645272,31.60177807&size=240arcsec&gzip=false in 5s
Retry 1/3 for https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci/2022/0220/510833/ztf_20220220510833_000679_zr_c08_o_q3_sciimg.fits?center=240.03559663,31.5785114&size=240arcsec&gzip=false in 5s
Retry 1/3 for https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci/2022/0219/536262/ztf_20220219536262_000679_zi_c08_o_q3_sciimg.fits?center=240.03645272,31.60177807&size=240arcsec&gzip=false in 5s
Retry 1/3 for https://irsa.ipac.caltech.

## 5. Download reference images

> Reference images are static coadds - they aren't tied to a single epoch the way science/diff images are. Every `df_meta` row for the same field/CCD/quadrant/filter points at the *same* reference image, so the cell below de-duplicates by filename before downloading.

> Reference images are inverted along the X axis compared to sci and diff.

In [22]:
ref_inputs = []
for _, row in df_meta.iterrows():
    base_ref_url = build_ref_url(row)
    original_fname = Path(urlparse(base_ref_url).path).name
    url = build_cutout_url(base_ref_url, row["in_ra"], row["in_dec"], size_arcsec=240)
    desired_fname = f"{row['gal_folder']}_RA{row['in_ra']:.4f}_DEC{row['in_dec']:.4f}_{row['filtercode']}__{original_fname}"

    ref_inputs.append((url, desired_fname, row['gal_folder']))

# Same field/ccd/quadrant/filter -> same reference image; drop the repeats
ref_inputs = list({fname: (url, fname, folder) for url, fname, folder in ref_inputs}.values())

batch_download_resumable(
    ref_inputs,
    out_dir="ref_test",
    progress_json="ref_test.progress.json",
    max_workers=9,
    chunk_size=200,
)

ref_cutouts = sorted(str(p) for p in Path("ref_test").rglob("*.fits*"))
print(f"Reference files currently on disk: {len(ref_cutouts)}")

Download chunk 1/2 (0:200) attempt 1
1 Downloaded: 6_RA240.0363_DEC31.3740_zr__ztf_001676_zr_c10_q1_refimg.fits
2 Downloaded: 1_RA240.0348_DEC31.5102_zr__ztf_001676_zr_c10_q1_refimg.fits
3 Downloaded: 5_RA240.0359_DEC31.4114_zr__ztf_001676_zr_c10_q1_refimg.fits
4 Downloaded: 4_RA240.0357_DEC31.3827_zr__ztf_001676_zr_c10_q1_refimg.fits
5 Downloaded: 3_RA240.0356_DEC31.5785_zi__ztf_000679_zi_c08_q3_refimg.fits
6 Downloaded: 8_RA240.0365_DEC31.6018_zi__ztf_000679_zi_c08_q3_refimg.fits
7 Downloaded: 2_RA240.0356_DEC31.4340_zr__ztf_001676_zr_c10_q1_refimg.fits
8 Downloaded: 10_RA240.0366_DEC31.4241_zr__ztf_001676_zr_c10_q1_refimg.fits
9 Downloaded: 1_RA240.0348_DEC31.5102_zr__ztf_000679_zr_c08_q3_refimg.fits
10 Downloaded: 7_RA240.0364_DEC31.4729_zr__ztf_001676_zr_c10_q1_refimg.fits
11 Downloaded: 1_RA240.0348_DEC31.5102_zi__ztf_000679_zi_c08_q3_refimg.fits
12 Downloaded: 8_RA240.0365_DEC31.6018_zr__ztf_000679_zr_c08_q3_refimg.fits
13 Downloaded: 3_RA240.0356_DEC31.5785_zr__ztf_000679_zr_c0